# Reddit — Data Preparation

## What this notebook does

The raw Reddit data was collected via the Reddit API and stored as **`.jsonl` files** (JSON Lines — one JSON record per line) in the Bronze folder, one file per subreddit. The analysis notebooks expect two **combined `.parquet` files**:

- `reddit_comments_raw.parquet` — all comments across all subreddits
- `reddit_posts_raw.parquet` — all posts across all subreddits

This notebook merges and converts them.

**Why parquet?** Parquet is a compressed columnar format. It is ~5–10× faster to load than JSON for large dataframes and uses significantly less disk space.

**Input:** `Data/1_Bronze/Reddit/r_*_comments.jsonl`, `r_*_posts.jsonl`  
**Output:** `Data/1_Bronze/Reddit/reddit_comments_raw.parquet`, `reddit_posts_raw.parquet`

## 0. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path

BRONZE = Path('../../Data/1_Bronze/Reddit')
print('Setup complete.')
print(f'Bronze folder: {BRONZE.resolve()}')

Setup complete.
Bronze folder: /Users/helenalips/Desktop/HIR 1ste master/Social Media and Web Analytics/group-project-SMWA/Data/1_Bronze/Reddit


## 1. Inspect available files

In [2]:
jsonl_files = sorted(BRONZE.glob('*.jsonl'))
print(f'Found {len(jsonl_files)} .jsonl files:')
for f in jsonl_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<45}  {size_mb:.1f} MB')

Found 14 .jsonl files:
  r_conservative_comments.jsonl                  1136.2 MB
  r_conservative_posts.jsonl                     157.9 MB
  r_democrats_comments.jsonl                     767.2 MB
  r_democrats_posts.jsonl                        88.2 MB
  r_liberal_comments.jsonl                       33.4 MB
  r_liberal_posts.jsonl                          6.3 MB
  r_politics_comments.jsonl                      3783.9 MB
  r_politics_posts.jsonl                         288.0 MB
  r_republican_comments.jsonl                    166.1 MB
  r_republican_posts.jsonl                       36.7 MB
  r_trump_comments.jsonl                         320.2 MB
  r_trump_posts.jsonl                            80.7 MB
  r_worldnews_comments.jsonl                     3228.1 MB
  r_worldnews_posts.jsonl                        147.8 MB


## 2. Convert and save to parquet incrementally

Instead of loading all files into memory at once, we process one file at a time and
append each batch directly to the parquet file using `pyarrow.ParquetWriter`.
This keeps memory usage low and avoids needing temporary disk space for large intermediate DataFrames.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Output folders — one parquet file per subreddit inside each folder
comments_dir = BRONZE / 'comments'
posts_dir    = BRONZE / 'posts'
comments_dir.mkdir(exist_ok=True)
posts_dir.mkdir(exist_ok=True)

def drop_nested_columns(df):
    """Drop columns containing dicts or lists — not parquet-compatible."""
    nested = [
        col for col in df.columns
        if df[col].dropna().apply(lambda x: isinstance(x, (dict, list))).any()
    ]
    return df.drop(columns=nested)

def convert_file(f, out_dir):
    df = pd.read_json(f, lines=True)
    df['subreddit'] = df['subreddit'].str.lower()
    if 'created_utc' in df.columns:
        df['created_utc'] = pd.to_datetime(df['created_utc'], unit='s', utc=True)
    df = drop_nested_columns(df)
    out_path = out_dir / f.name.replace('.jsonl', '.parquet')
    df.to_parquet(out_path, index=False, engine='pyarrow')
    print(f'  {f.name:<45}  {len(df):>7,} records  →  {out_path.name}')
    del df

print('Converting comments...')
for f in sorted(BRONZE.glob('*_comments.jsonl')):
    convert_file(f, comments_dir)

print('\nConverting posts...')
for f in sorted(BRONZE.glob('*_posts.jsonl')):
    convert_file(f, posts_dir)

print('\nDone.')
print(f'Comments folder: {comments_dir}')
print(f'Posts folder   : {posts_dir}')

Converting comments...
